<a href="https://colab.research.google.com/github/Debdeep17092002/CODSOFT_TASKNO/blob/main/moviegenreclassification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import argparse
import os
import time
import warnings

In [2]:
import joblib
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.naive_bayes import ComplementNB
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

In [3]:
warnings.filterwarnings("ignore", category=UserWarning)
SEP = " ::: "

#Data Helpers


In [4]:
def read_file(path, columns):
    with open(path, encoding="utf-8") as fh:
        rows = [line.rstrip("\n").split(SEP) for line in fh if line.strip()]
    rows = [r for r in rows if len(r) == len(columns)]
    return pd.DataFrame(rows, columns=columns)

In [5]:
def make_text(df):
    """Title carries signal too (e.g. 'Night of the ...'), so include it."""
    return (df["title"] + " . " + df["desc"]).str.lower()

In [6]:
def build_vectorizer():
    return TfidfVectorizer(
        ngram_range=(1, 2),     # unigrams + bigrams
        min_df=3,               # drop very rare terms
        max_df=0.8,             # drop near-ubiquitous terms
        max_features=60000,
        sublinear_tf=True,      # 1 + log(tf) dampens long descriptions
        stop_words="english",
    )

In [7]:
def candidate_models():
    return {
        "ComplementNB": ComplementNB(alpha=0.3),
        "LinearSVC (balanced)": LinearSVC(C=0.5, class_weight="balanced"),
        "LogisticRegression (balanced)": LogisticRegression(
            C=10, max_iter=200, class_weight="balanced"
        ),
    }

#Train/evaluate


In [8]:
def train(args):
    train_df = read_file(os.path.join(args.data_dir, "train_data.txt"),
                         ["id", "title", "genre", "desc"])
    test_df = read_file(os.path.join(args.data_dir, "test_data.txt"),
                        ["id", "title", "desc"])
    sol_path = os.path.join(args.data_dir, "test_data_solution.txt")
    sol_df = (read_file(sol_path, ["id", "title", "genre", "desc"])
              if os.path.exists(sol_path) else None)
    print(f"train={len(train_df):,}  test={len(test_df):,}  "
          f"genres={train_df.genre.nunique()}")

    t0 = time.time()
    vec = build_vectorizer()
    X_train = vec.fit_transform(make_text(train_df))
    X_test = vec.transform(make_text(test_df))
    print(f"TF-IDF matrix: {X_train.shape}  ({time.time() - t0:.0f}s)\n")

    y_train = train_df["genre"].values
    results, fitted = [], {}
    for name, model in candidate_models().items():
        t0 = time.time()
        model.fit(X_train, y_train)
        fitted[name] = model
        if sol_df is not None:
            pred = model.predict(X_test)
            y_true = sol_df["genre"].values
            results.append({
                "model": name,
                "accuracy": accuracy_score(y_true, pred),
                "macro_f1": f1_score(y_true, pred, average="macro"),
                "weighted_f1": f1_score(y_true, pred, average="weighted"),
                "fit_s": round(time.time() - t0),
            })
            print(f"{name:32s} acc={results[-1]['accuracy']:.4f}  "
                  f"macroF1={results[-1]['macro_f1']:.4f}  "
                  f"weightedF1={results[-1]['weighted_f1']:.4f}", flush=True)


    best = (max(results, key=lambda r: r["macro_f1"])["model"]
            if results else "LogisticRegression (balanced)")
    print(f"\nSelected model: {best}")
    pipe = Pipeline([("tfidf", vec), ("clf", fitted[best])])

    os.makedirs(args.out_dir, exist_ok=True)
    joblib.dump(pipe, os.path.join(args.out_dir, "genre_model.joblib"), compress=3)


    pred = pipe["clf"].predict(X_test)
    out = test_df.assign(genre=pred)[["id", "title", "genre", "desc"]]
    with open(os.path.join(args.out_dir, "test_predictions.txt"), "w",
              encoding="utf-8") as fh:
        for r in out.itertuples(index=False):
            fh.write(SEP.join(r) + "\n")

    if sol_df is not None:
        report = classification_report(sol_df["genre"].values, pred, zero_division=0)
        pd.DataFrame(results).to_csv(
            os.path.join(args.out_dir, "model_comparison.csv"), index=False)
        with open(os.path.join(args.out_dir, "classification_report.txt"), "w") as fh:
            fh.write(f"Model: {best}\n\n{report}")
        print("\n" + report)


In [9]:
def predict_one(args):
    pipe = joblib.load(args.model)
    df = pd.DataFrame({"title": [args.title], "desc": [args.predict]})
    clf = pipe["clf"]
    X = pipe["tfidf"].transform(make_text(df))
    print("Predicted genre:", clf.predict(X)[0])
    if hasattr(clf, "predict_proba"):
        proba = clf.predict_proba(X)[0]
        top = sorted(zip(clf.classes_, proba), key=lambda t: -t[1])[:3]
        print("Top 3:", ", ".join(f"{g} ({p:.0%})" for g, p in top))


In [11]:
if __name__ == "__main__":
    ap = argparse.ArgumentParser(description=__doc__,
                                 formatter_class=argparse.RawDescriptionHelpFormatter)
    default_data = next((p for p in ("/mnt/user-data/uploads", "/content", ".")
                         if os.path.exists(os.path.join(p, "train_data.txt"))), ".")
    ap.add_argument("--data-dir", default=default_data,
                    help="folder containing train_data.txt, test_data.txt, ...")
    ap.add_argument("--out-dir", default="outputs")
    ap.add_argument("--model", default="outputs/genre_model.joblib",
                    help="saved model, used with --predict")
    ap.add_argument("--predict", help="plot text to classify with a saved model")
    ap.add_argument("--title", default="", help="optional title for --predict")

    a, _ = ap.parse_known_args()
    predict_one(a) if a.predict else train(a)

train=54,214  test=54,200  genres=27
TF-IDF matrix: (54214, 60000)  (32s)

ComplementNB                     acc=0.5726  macroF1=0.2791  weightedF1=0.5175
LinearSVC (balanced)             acc=0.5754  macroF1=0.3991  weightedF1=0.5807
LogisticRegression (balanced)    acc=0.5970  macroF1=0.4224  weightedF1=0.5966

Selected model: LogisticRegression (balanced)

              precision    recall  f1-score   support

      action       0.41      0.51      0.45      1314
       adult       0.57      0.60      0.58       590
   adventure       0.38      0.32      0.35       775
   animation       0.35      0.25      0.29       498
   biography       0.11      0.04      0.06       264
      comedy       0.60      0.61      0.60      7446
       crime       0.23      0.21      0.22       505
 documentary       0.77      0.75      0.76     13096
       drama       0.67      0.62      0.64     13612
      family       0.29      0.28      0.29       783
     fantasy       0.23      0.18      0.20  